In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Auto-detect results directory
for candidate in ["../results/gsm8k", "../results_v2/gsm8k", "results/gsm8k", "results_v2/gsm8k"]:
    if os.path.isdir(candidate):
        RESULTS_DIR = candidate
        break
print(f"Results dir: {RESULTS_DIR}")

METHODS  = ["homo_r4", "homo_r8", "hetero_pad", "flexlora", "hetero_spa", "spa_m"]
SEEDS    = [42, 43, 44]
COLORS   = {"homo_r4": "#9E9E9E", "homo_r8": "#4CAF50", "hetero_pad": "#FF9800",
            "flexlora": "#2196F3", "hetero_spa": "#9C27B0", "spa_m": "#E91E63"}
LABELS   = {"homo_r4": "FedAvg r=4", "homo_r8": "FedAvg r=8", "hetero_pad": "Hetero-Pad",
            "flexlora": "FlexLoRA", "hetero_spa": "SPA", "spa_m": "SPA-M (Ours)"}

def load(method, seed):
    for alpha in ["05", "0.5", "00"]:
        for pattern in [
            f"{method}_seed{seed}_alpha{alpha}.json",
            f"{method}_seed{seed}_alpha0{alpha}.json",
        ]:
            path = os.path.join(RESULTS_DIR, pattern)
            if os.path.exists(path):
                with open(path) as f:
                    return json.load(f)
    # glob fallback
    matches = glob.glob(os.path.join(RESULTS_DIR, f"{method}_seed{seed}*.json"))
    if matches:
        with open(matches[0]) as f:
            return json.load(f)
    return None

data = {}
for m in METHODS:
    for s in SEEDS:
        d = load(m, s)
        if d:
            data[(m, s)] = d

print(f"Loaded {len(data)} runs")
for (m, s), d in sorted(data.items()):
    print(f"  {m:<12} seed={s}  rounds={len(d['rounds'])}")

In [ ]:
# Check what metric key is used
sample = next(iter(data.values()))
r0 = sample['rounds'][0]
METRIC_KEY = 'exact_match' if 'exact_match' in r0 else 'accuracy'
print(f"Metric key: {METRIC_KEY}")
print(f"Available keys: {list(r0.keys())}")

In [ ]:
# Summary table: Mean-L5, Final, Best
summary = {m: {"mean_l5": [], "final": [], "best": [], "seeds": []} for m in METHODS}

print(f"{'Method':<14} {'Seed':<6} {'Mean-L5★':>10} {'Final':>10} {'Best':>10}")
print("-" * 55)

for m in METHODS:
    for s in SEEDS:
        if (m, s) not in data:
            continue
        vals = [r[METRIC_KEY] for r in data[(m, s)]['rounds']]
        ml5  = float(np.mean(vals[-5:]))
        fin  = vals[-1]
        best = max(vals)
        summary[m]['mean_l5'].append(ml5)
        summary[m]['final'].append(fin)
        summary[m]['best'].append(best)
        summary[m]['seeds'].append(s)
        print(f"{m:<14} {s:<6} {ml5*100:>9.2f}% {fin*100:>9.2f}% {best*100:>9.2f}%")
    print()

print("=" * 55)
print(f"{'Method':<14} {'n':<4} {'Mean-L5★':>14} {'Final':>14} {'Best':>14}")
print("-" * 62)
for m in METHODS:
    ml5s = summary[m]['mean_l5']
    fins = summary[m]['final']
    bsts = summary[m]['best']
    if not ml5s:
        continue
    n = len(ml5s)
    print(f"{m:<14} {n:<4} "
          f"{np.mean(ml5s)*100:>6.2f}±{np.std(ml5s)*100:.2f}%  "
          f"{np.mean(fins)*100:>6.2f}±{np.std(fins)*100:.2f}%  "
          f"{np.mean(bsts)*100:>6.2f}±{np.std(bsts)*100:.2f}%")

In [ ]:
# Mean ± std convergence curves
methods_with_data = [m for m in METHODS if summary[m]['mean_l5']]
per_round = {m: [] for m in methods_with_data}
for m in methods_with_data:
    for s in SEEDS:
        if (m, s) not in data:
            continue
        per_round[m].append([r[METRIC_KEY] for r in data[(m, s)]['rounds']])

fig, ax = plt.subplots(figsize=(10, 5))
for m in methods_with_data:
    arr = np.array(per_round[m])
    mu  = arr.mean(axis=0)
    std = arr.std(axis=0)
    x   = np.arange(1, len(mu) + 1)
    n   = len(arr)
    ax.plot(x, mu * 100, label=f"{LABELS[m]} (n={n})", color=COLORS[m], linewidth=2)
    ax.fill_between(x, (mu - std) * 100, (mu + std) * 100, alpha=0.15, color=COLORS[m])

ax.set_xlabel("Communication Round", fontsize=12)
ax.set_ylabel("Exact Match Accuracy (%)", fontsize=12)
ax.set_title("GSM8K — Math Reasoning | Qwen2.5-7B | α=0.5", fontsize=13)
ax.legend(fontsize=9, loc="lower right")
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.savefig("gsm8k_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bar chart: Mean-L5 with error bars
methods_with_data = [m for m in METHODS if summary[m]['mean_l5']]
means = [np.mean(summary[m]['mean_l5']) * 100 for m in methods_with_data]
stds  = [np.std(summary[m]['mean_l5']) * 100  for m in methods_with_data]
colors = [COLORS[m] for m in methods_with_data]
labels = [LABELS[m] for m in methods_with_data]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.set_ylabel("Mean-L5 Exact Match (%)")
ax.set_title("GSM8K Summary — Mean-L5 Accuracy ★")
ax.set_ylim(0, max(means) * 1.2)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig("gsm8k_bar.png", dpi=150, bbox_inches="tight")
plt.show()